In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import string

In [6]:
import nltk

In [7]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shubh\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [9]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\shubh\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

In [3]:
df = pd.read_csv("data/cfpb-complaints-2026-02-05_20_10.csv",engine="python",
    on_bad_lines="skip")
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,01/20/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I am writing to have the following information...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,92345,NaN,Consent provided,Web,01/20/25,Closed with non-monetary relief,Yes,NaN,11588109
1,07/03/24,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,I am a victim of identity theft. Please delete...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,32824,NaN,Consent provided,Web,07/03/24,Closed with non-monetary relief,Yes,NaN,9416677
2,09/14/25,Vehicle loan or lease,Loan,Incorrect information on your report,Information belongs to someone else,"My name is XXXX XXXX, and I am formally disput...",NaN,"SANTANDER HOLDINGS USA, INC.",PA,19143,NaN,Consent provided,Web,09/14/25,Closed with explanation,Yes,NaN,15930829
3,05/01/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"Upon reviewing my credit report, I have identi...",Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,76105,NaN,Consent provided,Web,05/01/25,Closed with non-monetary relief,Yes,NaN,13274568
4,12/08/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,Everything is explained in my resolution packa...,Company believes it acted appropriately as aut...,Kubota North America Corporation,MS,391XX,NaN,Consent provided,Web,12/08/25,Closed with explanation,Yes,NaN,17837761


In [4]:
class TextPreprocessor:
    """
    Comprehensive text preprocessing for complaint narratives
    """
    
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        # Keep some important words that are usually stopwords
        self.stop_words -= {'not', 'no', 'never', 'neither', 'nor', 'nothing'}
        
    def clean_text(self, text):
        """Clean and normalize text"""
        if pd.isna(text):
            return ""
        
        # Convert to lowercase
        text = text.lower()
        
        # Remove URLs
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        
        # Remove email addresses
        text = re.sub(r'\S+@\S+', '', text)
        
        # Remove phone numbers (various formats)
        text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '', text)
        text = re.sub(r'\(\d{3}\)\s*\d{3}[-.]?\d{4}', '', text)
        
        # Remove account numbers (sequences of 8+ digits)
        text = re.sub(r'\b\d{8,}\b', '[ACCOUNT_NUM]', text)
        
        # Remove dollar amounts
        text = re.sub(r'\$\s*\d+[,\d]*\.?\d*', '[AMOUNT]', text)
        
        # Remove extra whitespace
        text = ' '.join(text.split())
        
        return text
    
    def remove_stopwords(self, text):
        """Remove stopwords while preserving negations"""
        tokens = word_tokenize(text)
        filtered = [word for word in tokens if word not in self.stop_words and word not in string.punctuation]
        return ' '.join(filtered)
    
    def preprocess(self, text):
        """Full preprocessing pipeline"""
        text = self.clean_text(text)
        # Keeping stopwords for sentiment analysis
        # Only remove for topic modeling
        return text

In [10]:
def preprocess_complaints(df):
    """
    Preprocess all complaint narratives
    
    Parameters:
    -----------
    df : pandas.DataFrame
        CFPB complaints dataframe
        
    Returns:
    --------
    df : pandas.DataFrame
        Dataframe with preprocessed text columns
    """
    
    print("=" * 70)
    print("TEXT PREPROCESSING")
    print("=" * 70)
    
    preprocessor = TextPreprocessor()
    
    print("\nPreprocessing narratives...")
    
    # Create cleaned text column
    df['narrative_clean'] = df['Consumer complaint narrative'].apply(preprocessor.preprocess)
    
    # Create version without stopwords (for topic modeling)
    df['narrative_no_stopwords'] = df['narrative_clean'].apply(preprocessor.remove_stopwords)
    
    # Calculate word count
    df['word_count'] = df['narrative_clean'].apply(lambda x: len(x.split()))
    
    print(f"Preprocessed {len(df)} complaints")
    
    # Show example
    print("\nExample preprocessing:")
    idx = df[df['Consumer complaint narrative'].notna()].index[0]
    
    print(f"\n  ORIGINAL ({len(df.loc[idx, 'Consumer complaint narrative'])} chars):")
    print(f"  {df.loc[idx, 'Consumer complaint narrative'][:300]}...")
    
    print(f"\n  CLEANED ({len(df.loc[idx, 'narrative_clean'])} chars):")
    print(f"  {df.loc[idx, 'narrative_clean'][:300]}...")
    
    # Statistics
    print("\nPreprocessing statistics:")
    print(f"  Average word count: {df['word_count'].mean():.2f}")
    print(f"  Median word count:  {df['word_count'].median():.2f}")
    print(f"  Min word count:     {df['word_count'].min():.0f}")
    print(f"  Max word count:     {df['word_count'].max():.0f}")
    
    return df


# ─── USAGE ───
df = preprocess_complaints(df)

TEXT PREPROCESSING

Preprocessing narratives...
Preprocessed 1400010 complaints

Example preprocessing:

  ORIGINAL (662 chars):
  I am writing to have the following information removed from my credit file, the items that I need deleted are going to be attached in a word document. I am a victim of identity theft. I have multiple accounts and inquiries that I DID NOT apply for listed on my credit report. I ask that these items b...

  CLEANED (662 chars):
  i am writing to have the following information removed from my credit file, the items that i need deleted are going to be attached in a word document. i am a victim of identity theft. i have multiple accounts and inquiries that i did not apply for listed on my credit report. i ask that these items b...

Preprocessing statistics:
  Average word count: 173.54
  Median word count:  120.00
  Min word count:     1
  Max word count:     6469


In [11]:
df.to_csv("data/text_processed_complaints.csv")

In [12]:
df.shape

(1400010, 21)

In [13]:
df.head(5)

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,...,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,narrative_clean,narrative_no_stopwords,word_count
0,01/20/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I am writing to have the following information...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,92345,...,Consent provided,Web,01/20/25,Closed with non-monetary relief,Yes,NaN,11588109,i am writing to have the following information...,writing following information removed credit f...,119
1,07/03/24,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,I am a victim of identity theft. Please delete...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,32824,...,Consent provided,Web,07/03/24,Closed with non-monetary relief,Yes,NaN,9416677,i am a victim of identity theft. please delete...,victim identity theft please delete remove ite...,68
2,09/14/25,Vehicle loan or lease,Loan,Incorrect information on your report,Information belongs to someone else,"My name is XXXX XXXX, and I am formally disput...",NaN,"SANTANDER HOLDINGS USA, INC.",PA,19143,...,Consent provided,Web,09/14/25,Closed with explanation,Yes,NaN,15930829,"my name is xxxx xxxx, and i am formally disput...",name xxxx xxxx formally disputing fraudulent a...,140
3,05/01/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"Upon reviewing my credit report, I have identi...",Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,76105,...,Consent provided,Web,05/01/25,Closed with non-monetary relief,Yes,NaN,13274568,"upon reviewing my credit report, i have identi...",upon reviewing credit report identified inaccu...,18
4,12/08/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,Everything is explained in my resolution packa...,Company believes it acted appropriately as aut...,Kubota North America Corporation,MS,391XX,...,Consent provided,Web,12/08/25,Closed with explanation,Yes,NaN,17837761,everything is explained in my resolution packa...,everything explained resolution package ive al...,17
